In [1]:
import pandas as pd

# Load data
df = pd.read_csv('articles_479.csv')

# List target keywords
all_keywords = ['epigenetic', 'epigenetics', 'regulation', 'mechanism', 'tree', 'trees', 'woody plants', 'plant', 'plants']

def calculate_relevance(abstract):
    if pd.isna(abstract): 
        return 0
    abstract = abstract.lower()
    # Count how many keywords are found
    score = sum(1 for word in all_keywords if word in abstract)
    return score

# Apply scoring
df['relevance_score'] = df['abstract'].apply(calculate_relevance)

# Sort by highest score
df_sorted = df.sort_values(by='relevance_score', ascending=False)

# Save sorted results as CSV
df_sorted.to_csv('sorted_screening_479.csv', index=False)

# Also save as RIS format
def save_ris(dataframe, filepath):
    """Convert DataFrame to RIS format"""
    with open(filepath, 'w', encoding='utf-8') as f:
        for idx, row in dataframe.iterrows():
            f.write(f"TI  - {row.get('title', 'Unknown')}\n")
            if pd.notna(row.get('abstract')):
                f.write(f"AB  - {row['abstract']}\n")
            if pd.notna(row.get('author')):
                f.write(f"AU  - {row.get('author')}\n")
            if pd.notna(row.get('year')):
                f.write(f"PY  - {row.get('year')}\n")
            if pd.notna(row.get('journal')):
                f.write(f"SO  - {row.get('journal')}\n")
            # Add relevance score
            f.write(f"N1  - Relevance Score: {row['relevance_score']}\n")
            f.write("ER  - \n\n")

save_ris(df_sorted, 'sorted_screening_479.ris')

# Display results
print(f"Total records: {len(df)}")
print(f"Records with score > 0: {(df_sorted['relevance_score'] > 0).sum()}")
print(f"\n🏆 Top 5 Most Relevant Papers:")
print(df_sorted[['title', 'relevance_score']].head())

print(f"\n✅ CSV saved to: sorted_screening_479.csv")
print(f"✅ RIS saved to: sorted_screening_479.ris")

Total records: 479
Records with score > 0: 475

🏆 Top 5 Most Relevant Papers:
                                                 title  relevance_score
94   Effects of ploidy variation on DNA methylation...                9
75   Vascular Cambium—Between the Hammer and the An...                8
378  Small RNA-Sequencing Links Physiological Chang...                8
400  MTA, an RNA m(6)A Methyltransferase, Enhances ...                8
50   Can DNA methylation shape climate response in ...                8

✅ CSV saved to: sorted_screening_479.csv
✅ RIS saved to: sorted_screening_479.ris


In [2]:
import pandas as pd
import re

# Load CSV data
df = pd.read_csv('articles_479.csv')

print("="*80)
print("RELEVANCE SCORING - CSV VERSION")
print("="*80)

print(f"\nTotal records loaded: {len(df)}")
print(f"Columns: {list(df.columns)}")

# Define keyword categories with weights
keyword_categories = {
    'epigenetics': {
        'keywords': ['epigenetic', 'epigenetics', 'histone', 'dna methylation', 'chromatin'],
        'weight': 3
    },
    'regulation': {
        'keywords': ['regulation', 'regulatory', 'gene expression', 'transcription'],
        'weight': 2
    },
    'mechanism': {
        'keywords': ['mechanism', 'pathway', 'process', 'signaling'],
        'weight': 2
    },
    'plant_type': {
        'keywords': ['tree', 'trees', 'woody plants', 'forest', 'woody'],
        'weight': 2
    },
    'general_plant': {
        'keywords': ['plant', 'plants', 'vegetation', 'flora'],
        'weight': 1
    }
}

def calculate_relevance_advanced(text):
    """
    Advanced relevance scoring with:
    - Word boundary matching (whole words only)
    - Category-based weighting
    - Bonus for multi-category matches
    """
    if pd.isna(text) or text == '':
        return 0
    
    text = text.lower()
    total_score = 0
    matched_categories = set()
    
    for category, data in keyword_categories.items():
        keywords = data['keywords']
        weight = data['weight']
        
        for keyword in keywords:
            # Use word boundaries to avoid partial matches
            pattern = r'\b' + re.escape(keyword) + r'\b'
            
            if re.search(pattern, text):
                total_score += weight
                matched_categories.add(category)
    
    # Bonus for combining multiple categories (50% boost)
    if len(matched_categories) >= 2:
        total_score *= 1.5
    
    return round(total_score, 2)

def calculate_relevance_with_details(text):
    """Return score and matched keywords for debugging"""
    if pd.isna(text) or text == '':
        return 0, [], []
    
    text = text.lower()
    total_score = 0
    matched_keywords = []
    matched_categories = set()
    
    for category, data in keyword_categories.items():
        keywords = data['keywords']
        weight = data['weight']
        
        for keyword in keywords:
            pattern = r'\b' + re.escape(keyword) + r'\b'
            if re.search(pattern, text):
                total_score += weight
                matched_keywords.append((keyword, category, weight))
                matched_categories.add(category)
    
    bonus_multipliers = []
    
    # Apply multi-category bonus
    if len(matched_categories) >= 2:
        multiplier = 1.5
        bonus_multipliers.append(('Multi-Category Bonus', multiplier))
        total_score *= multiplier
    
    return round(total_score, 2), matched_keywords, bonus_multipliers

# Apply scoring
df['relevance_score'] = df['abstract'].apply(calculate_relevance_advanced)

# Sort by highest score
df_sorted = df.sort_values(by='relevance_score', ascending=False)

# Create detailed results list
detailed_results = []

for rank, (idx, row) in enumerate(df_sorted.iterrows(), 1):
    score, keywords, bonuses = calculate_relevance_with_details(row['abstract'])
    
    # Format keywords found
    keywords_found = []
    for kw, cat, weight in keywords:
        keywords_found.append(f"{kw}({cat}, w:{weight})")
    keywords_str = " | ".join(keywords_found) if keywords_found else "None"
    
    # Format bonuses
    bonuses_str = " | ".join([f"{b[0]}({b[1]}x)" for b in bonuses]) if bonuses else "None"
    
    detailed_results.append({
        'Rank': rank,
        'Score': score,
        'Title': row.get('title', 'Unknown'),
        'Author': row.get('author', 'Unknown'),
        'Year': row.get('year', 'Unknown'),
        'Journal': row.get('journal', 'Unknown'),
        'Keywords_Found': keywords_str,
        'Bonuses_Applied': bonuses_str,
        'Abstract': row.get('abstract', 'Unknown')
    })

# Create DataFrame from detailed results
df_detailed = pd.DataFrame(detailed_results)

# Display results
print(f"\n{'='*80}")
print(f"SCORING RESULTS")
print(f"{'='*80}")
print(f"\nTotal records: {len(df)}")
print(f"Records with score > 0: {(df_sorted['relevance_score'] > 0).sum()}")
print(f"Max score: {df_sorted['relevance_score'].max()}")
print(f"Mean score (non-zero): {df_sorted[df_sorted['relevance_score'] > 0]['relevance_score'].mean():.2f}")

print(f"\n{'='*80}")
print(f"TOP 15 MOST RELEVANT RECORDS")
print(f"{'='*80}\n")

for rank, (idx, row) in enumerate(df_sorted.head(15).iterrows(), 1):
    score, keywords, bonuses = calculate_relevance_with_details(row['abstract'])
    print(f"{rank}. Score: {score} | {row.get('Title', 'N/A')[:70]}")
    
    if keywords:
        categories_matched = {}
        for kw, cat, weight in keywords:
            if cat not in categories_matched:
                categories_matched[cat] = []
            categories_matched[cat].append(f"{kw}(w:{weight})")
        
        for cat, kws in categories_matched.items():
            print(f"      └─ {cat}: {', '.join(kws)}")
    
    if bonuses:
        bonus_text = " + ".join([f"{b[0]}({b[1]}x)" for b in bonuses])
        print(f"      🎯 Bonuses: {bonus_text}")
    
    print(f"      Abstract: {str(row.get('abstract', 'N/A'))[:100]}...\n")

# Save detailed results as CSV (all records sorted by score)
output_csv_detailed = 'sorted_screening_detailed_results.csv'
df_detailed.to_csv(output_csv_detailed, index=False)
print(f"✅ Detailed results (all records) saved to: {output_csv_detailed}")

# Save sorted results as CSV (original columns + score)
output_csv = 'sorted_screening_all.csv'
df_sorted.to_csv(output_csv, index=False)
print(f"✅ All sorted records saved to: {output_csv}")

# Convert to RIS format
def save_ris(dataframe, filepath):
    """Convert DataFrame to RIS format"""
    with open(filepath, 'w', encoding='utf-8') as f:
        for idx, row in dataframe.iterrows():
            f.write(f"TI  - {row.get('title', 'Unknown')}\n")
            if pd.notna(row.get('abstract')):
                f.write(f"AB  - {row['abstract']}\n")
            if pd.notna(row.get('author')):
                f.write(f"AU  - {row.get('author')}\n")
            if pd.notna(row.get('year')):
                f.write(f"PY  - {row.get('year')}\n")
            if pd.notna(row.get('journal')):
                f.write(f"SO  - {row.get('journal')}\n")
            # Add relevance score
            f.write(f"N1  - Relevance Score: {row['relevance_score']}\n")
            f.write("ER  - \n\n")

output_ris = 'sorted_screening_all.ris'
save_ris(df_sorted, output_ris)
print(f"✅ RIS results saved to: {output_ris}")

print("\n" + "="*80)

RELEVANCE SCORING - CSV VERSION

Total records loaded: 479
Columns: ['key', 'title', 'year', 'month', 'day', 'journal', 'issn', 'volume', 'issue', 'pages', 'authors', 'url', 'language', 'publisher', 'location', 'abstract', 'notes', 'doi', 'keywords', 'pubmed_id', 'pmc_id']

SCORING RESULTS

Total records: 479
Records with score > 0: 475
Max score: 36.0
Mean score (non-zero): 17.52

TOP 15 MOST RELEVANT RECORDS

1. Score: 36.0 | N/A
      └─ epigenetics: epigenetic(w:3), histone(w:3), dna methylation(w:3)
      └─ regulation: regulation(w:2), gene expression(w:2)
      └─ plant_type: tree(w:2), trees(w:2), woody plants(w:2), forest(w:2), woody(w:2)
      └─ general_plant: plants(w:1)
      🎯 Bonuses: Multi-Category Bonus(1.5x)
      Abstract: In trees, the annual cycling of active and dormant states in buds is closely regulated by environmen...

2. Score: 34.5 | N/A
      └─ epigenetics: epigenetic(w:3), histone(w:3), dna methylation(w:3)
      └─ regulation: regulation(w:2), regulatory